### Imports

In [1]:
import pandas as pd
from swing_high_low_detection import swing_highs_lows_online

### Load CSV (safe) + quick preview

In [2]:
ohlc_clean = pd.read_parquet('ETHUSDT_15m_ohlc_clean.parquet')

ohlc_clean.head()

,timestamp,open,high,low,close,volume,segment_id
0,2021-07-05 12:00:00+00:00,2196.61,2214.83,2188.94,2208.84,4.2843,0
1,2021-07-05 12:15:00+00:00,2208.84,2219.13,2208.56,2211.99,0.0128,0
2,2021-07-05 12:30:00+00:00,2211.99,2218.00,2209.42,2214.01,0.0137,0
3,2021-07-05 12:45:00+00:00,2214.01,2214.90,2209.76,2210.56,0.0075,0
4,2021-07-05 13:00:00+00:00,2210.56,2226.37,2208.64,2224.07,0.0137,0


### Run the swing detection
#### Produces swings with 'HighLow' = 1 for highs, -1 for lows, NaN otherwise

label_type: rule_swing
N_candidates: [5, 10, 20, 50]
N_confirmation: 3
min_move_threshold: 0.0
min_bars_between_swings: 3
uses_future: true
computed_on: ETHUSDT_15m_ohlc_clean.parquet


In [3]:
swings_df = swing_highs_lows_online(ohlc_clean[["high", "low", "close"]])

### Build labels DataFrame
#### 0/1 encoding for ML: 1 = swing, 0 = no swing

In [4]:
labels_df = pd.DataFrame({
    "timestamp": ohlc_clean["timestamp"],
    "segment_id": ohlc_clean["segment_id"],
    "y_high_rule": (swings_df["HighLow"] == 1).astype(int),
    "y_low_rule":  (swings_df["HighLow"] == -1).astype(int),
})

### Quick sanity check: how many swings were detected and how df looks like

In [5]:
print(labels_df[["y_high_rule", "y_low_rule"]].sum())

y_high_rule    16358
y_low_rule     15810
dtype: int64


In [6]:
labels_df.head()

,timestamp,segment_id,y_high_rule,y_low_rule
0,2021-07-05 12:00:00+00:00,0,1,0
1,2021-07-05 12:15:00+00:00,0,0,0
2,2021-07-05 12:30:00+00:00,0,0,0
3,2021-07-05 12:45:00+00:00,0,0,0
4,2021-07-05 13:00:00+00:00,0,1,0


### Save labels as a separate file

In [7]:
labels_df.to_parquet("swing_labels.parquet",index=False)